# 01 Ingest Exploration — Agent 01 (Data & Question Registry)

> **Run ID**: `run_20261111_002`  
> **Contract**: `answers_normalized_v1` (`data/normalized/answers_normalized.parquet`)  
> **Raw Source**: `data/raw/survey_raw.csv` (SHA-256: `56acded3e02be49be27d6b288fa2c43fcce68df24525113eba1b1161eec52a95`)  

This exploratory notebook verifies raw survey immutability, executes the RFC-compliant ingestion parser, demonstrates explicit missingness preservation, and inspects the 13 substantive `QuestionSpec` configurations.

In [ ]:
import os
import sys
from pathlib import Path
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "data").is_dir()), Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
import pandas as pd
import json
from src.ingest.ingestion import SurveyIngester, compute_sha256, EXPECTED_RAW_SHA256
from src.registry.question_registry import QuestionRegistry
from src.hardcoding_scanner import verify_gate_1

# 1. Verify Raw Source Immutability
raw_csv = Path('data/raw/survey_raw.csv')
raw_sha = compute_sha256(raw_csv)
print(f"Raw CSV SHA-256: {raw_sha}")
assert raw_sha == EXPECTED_RAW_SHA256, "Raw CSV hash mismatch!"
print("Raw source cryptographic integrity VERIFIED.")

In [ ]:
# 2. Ingest & Normalize Raw Survey Data
ingester = SurveyIngester(raw_csv_path=raw_csv)
df_answers = ingester.normalize()
p_path, j_path = ingester.export(df_answers)

print(f"Total Normalized Answer Records: {len(df_answers)}")
print(f"Unique Participants: {df_answers['participant_id'].nunique()}")
print(f"Unique Questions: {df_answers['question_id'].nunique()}")
print("\nMissingness Breakdown:")
print(df_answers['missingness_state'].value_counts())

In [ ]:
# 3. Build & Inspect 13-Question Registry
registry = QuestionRegistry(raw_csv_path=raw_csv)
specs = registry.build_from_answers(df_answers)
registry.save_registry('data/normalized/question_registry.json')

substantive_specs = registry.get_substantive_questions()
print(f"Registered {len(specs)} questions total ({len(substantive_specs)} substantive).\n")
for s in substantive_specs:
    print(f"[{s.question_id}] {s.canonical_prompt_en[:45]}...")
    print(f"    Frame: {s.analytical_frame.value} | Economic Good Policy: {s.economic_good_classification_policy}")
    print(f"    Observed Usable: {s.observed_response_count} | Missing: {s.missing_response_count}")
    if s.missing_participant_ids:
        print(f"    Missing IDs: {s.missing_participant_ids}")

In [ ]:
# 4. Execute Agent 11 Quality Gate 1 Verification
gate_report = verify_gate_1()
print(json.dumps(gate_report, indent=2))
assert gate_report["status"] == "PASS", "Gate 1 check failed!"
print("\n>>> GATE 1 VERIFICATION PASSED <<<")